# 1. Overture ↔ Foursquare positive pairs (Connecticut)

Builds positive address pairs for CT: the same place's address as published by Overture Maps (side A) vs. the Foursquare Open Source Places record that Overture's bridge files link it to (side B). DuckDB, reading the public releases directly.

**Releases:** Overture Maps Places + bridge files `2026-09-23.1`, Foursquare OS Places `2026-09-15`.

**Stages**
1. `ct_places`: Overture places in CT
2. `ct_bridge`: GERS id → Foursquare / AllThePlaces record ids (same release)
3. `fsq_ct`: Foursquare OS Places records around CT
4. `ct_positives`: the 3-way join → positive pairs
5. What's in the positives
6. `ct_positives_clean`: address cleaning (units, floors, punctuation) + `same_street_address` flag
7. `ct_positives_flagged`: `number_mismatch` / `number_missing` flags, drop NULL / empty addresses
8. ZIP health check
9. `ct_positives_zip_cleaned`: `zip_5` (side A, the split key) and `zip_b` (side B)

`02_build_pairs.ipynb` picks up from `ct_positives_zip_cleaned`.

⚠️ Cells marked **RUN ONCE** download from S3 / Hugging Face and take minutes (they skip themselves if the output exists). Foursquare OS Places is gated on Hugging Face: accept its terms and set `HF_TOKEN` in your environment. All outputs are written to `Data/` (git-ignored).

In [ ]:
import os
import duckdb

OVERTURE_RELEASE = "2026-09-23.1"   # Overture places + bridge files release; also in every output filename
FSQ_RELEASE = "2026-09-15"        # Foursquare OS Places snapshot

con = duckdb.connect()                     # in-memory; tables vanish when the kernel restarts
con.sql("INSTALL httpfs; LOAD httpfs;")    # lets DuckDB read s3:// and https:// paths
con.sql("SET s3_region = 'us-west-2';")    # Overture's public bucket region

## 1. Overture places in CT — `ct_places`

- Bounding box around CT for speed (lets DuckDB skip row groups), then an exact CT filter.
- `region` values were `CT` and `ct` → compare with `UPPER()`.
- 393 places with NULL `region` but a CT ZIP (`06…`) are kept; the rest of the NULL-region rows are NY/MA/RI or have no postcode.
- `confidence` (does the place exist?) is kept as a column, not filtered — it says nothing about address quality.

Result: **193,270 places**, all ids unique.

In [ ]:
# RUN ONCE (skips itself if the output file exists) — scans Overture places on S3 (~11 GB total, but only CT row groups/columns are fetched)
out_path = f'Data/ct_places_{OVERTURE_RELEASE}.parquet'
if os.path.exists(out_path):
    print(f"skip: {out_path} already exists")
else:
    con.sql(f"""
    CREATE TABLE ct_places AS
    SELECT id, names.primary AS name, UPPER(addresses[1].freeform) AS freeform, UPPER(addresses[1].locality) AS locality, addresses[1].postcode AS postcode, addresses[1].region AS region, confidence, sources, bbox
    FROM read_parquet('s3://overturemaps-us-west-2/release/{OVERTURE_RELEASE}/theme=places/type=place/*.zstd.parquet')
    WHERE addresses[1].freeform IS NOT NULL
    AND
        bbox.xmin BETWEEN -73.73 AND -71.78 AND
        bbox.ymin BETWEEN 40.98 AND 42.06 AND
        -- parentheses matter: AND binds tighter than OR
        (UPPER(addresses[1].region) = 'CT'
        OR (addresses[1].region IS NULL AND addresses[1].postcode LIKE '06%'))
    """)

    con.sql(f"""
        COPY ct_places TO
        'Data/ct_places_{OVERTURE_RELEASE}.parquet' (FORMAT parquet)
    """)

## 2. Bridge links — `ct_bridge`

Overture's published record keeps only the provider that "won" the merge (`sources[1]`, verified `property = ''` for every row). The **bridge files** list every provider record that was merged into each GERS id — that's what turns single records into pairs.

- `provider` = whose record this bridge row points to (side B candidate; comes from the `provider=…` folder name)
- `published_from` = whose address Overture published (side A)
- `provider = published_from` → **self-pair** (same text on both sides) → not usable

Result: 58,694 rows (50,181 Foursquare + 8,513 AllThePlaces, one record per provider per place). Self-pairs: 12,483 FSQ + 2,067 ATP → **44,144 usable**.

In [ ]:
# RUN ONCE (skips itself if the output file exists) — scans the FSQ (~800 MB) and ATP (~210 MB) bridge files on S3
# UNION ALL stacks the two results; columns are matched by POSITION, so both SELECTs must list them in the same order
out_path = f'Data/ct_bridge_{OVERTURE_RELEASE}.parquet'
if os.path.exists(out_path):
    print(f"skip: {out_path} already exists")
else:
    con.sql(f"""
        CREATE TABLE ct_bridge AS 

        SELECT p.id AS gers_id, b.record_id, b.provider, LOWER(p.sources[1].dataset) AS published_from
        FROM read_parquet('Data/ct_places_{OVERTURE_RELEASE}.parquet') AS p
        JOIN read_parquet('s3://overturemaps-us-west-2/bridgefiles/{OVERTURE_RELEASE}/provider=foursquare/theme=places/type=place/*.parquet') AS b
        ON p.id = b.id

        UNION ALL

        SELECT p.id AS gers_id, b.record_id, b.provider, LOWER(p.sources[1].dataset) AS published_from
        FROM read_parquet('Data/ct_places_{OVERTURE_RELEASE}.parquet') AS p
        JOIN read_parquet('s3://overturemaps-us-west-2/bridgefiles/{OVERTURE_RELEASE}/provider=alltheplaces/theme=places/type=place/*.parquet') AS b
        ON p.id = b.id;

        COPY ct_bridge TO 'Data/ct_bridge_{OVERTURE_RELEASE}.parquet' (FORMAT parquet)
    """)

## 3. Foursquare records around CT — `fsq_ct`

Foursquare OS Places is gated on Hugging Face: the token comes from the `HF_TOKEN` environment variable. Never print it.

- **Bounding box only, no region filter.** Some Foursquare places located in CT carry wrong states (`KS`, `CA`, …); a region filter would drop them and lose their pairs. CT-ness was already decided on the Overture side — this table is just a lookup, and the join does the exact selection.
- `region` is kept raw (`CT` / `Connecticut` / `ct` …) — normalize it in feature code, not here.

Coverage check: only **31 of 50,181** bridge Foursquare ids are missing from `fsq_ct` (99.94%) — the id formats match.

In [ ]:
# Register the Hugging Face token with DuckDB (temporary: lasts for this connection only)
con.sql(f"CREATE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

In [ ]:
# RUN ONCE (skips itself if the output file exists) — reads Foursquare OS Places from Hugging Face
out_path = f'Data/fsq_ct_{FSQ_RELEASE}.parquet'
if os.path.exists(out_path):
    print(f"skip: {out_path} already exists")
else:
    con.sql(f"""
        CREATE TABLE fsq_ct AS 
        SELECT 
        fsq_place_id, name,
        UPPER(address) AS address, 
        UPPER(locality) AS locality,
        region, postcode,
        latitude, longitude, bbox,
        date_closed
        FROM 
        read_parquet('hf://datasets/foursquare/fsq-os-places/release/dt={FSQ_RELEASE}/places/parquet/*.parquet')
        WHERE 
        bbox.xmin BETWEEN -73.73 AND -71.78 AND
        bbox.ymin BETWEEN 40.98 AND 42.06 
    """
    )

    con.sql(f"""
        COPY fsq_ct
        TO 'Data/fsq_ct_{FSQ_RELEASE}.parquet' (FORMAT PARQUET)
    """)

## 4. Positive pairs — `ct_positives`

Chain of joins through the bridge (the middle table): `places ── gers_id ── bridge ── record_id ── foursquare`.
Foursquare only (AllThePlaces records are not used). Self-pairs excluded.

Result: **37,680 pairs**. `identical_address` flags the 70% (26,286) whose strings are exactly equal — kept, but report results on the non-identical subset.

In [ ]:
con.sql(f"""
    DROP TABLE IF EXISTS ct_positives;
    CREATE TABLE ct_positives AS 
    SELECT 
    b.gers_id, f.fsq_place_id,            -- ids kept for tracing pairs in error analysis
    p.freeform AS address_a,              -- _a = Overture (published_from provider)
    f.address AS address_b,               -- _b = Foursquare
    p.locality AS locality_a,
    f.locality AS locality_b,
    p.region AS region_a,
    f.region AS region_b,
    p.postcode AS postcode_a,
    f.postcode AS postcode_b,
    p.name AS name_a,                     -- names are NOT features (address-only scope); for reading pairs only
    f.name AS name_b,
    b.published_from,
    1 AS label,
    address_a = address_b AS identical_address
    FROM read_parquet('Data/ct_bridge_{OVERTURE_RELEASE}.parquet') AS b
    JOIN read_parquet('Data/ct_places_{OVERTURE_RELEASE}.parquet') AS p 
    ON b.gers_id = p.id
    JOIN read_parquet('Data/fsq_ct_{FSQ_RELEASE}.parquet') AS f
    on b.record_id = f.fsq_place_id
    WHERE b.provider = 'foursquare' AND b.provider <> b.published_from;

    COPY ct_positives TO 'Data/ct_positives_{OVERTURE_RELEASE}.parquet'
    (FORMAT parquet);
""")

## 5. What's in the positives

Eyeball sample of the non-identical pairs. Categories seen: unit/suite added on one side, abbreviations, ranges (`16-18`), ZIP+4, junk (`\N\N`), real typos (`DANBURY EOAD`), locality variants (`SPRAGUE`/`BALTIC`), and different house numbers.

House-number comparison. Result: 1,177 missing a number on one side (ambiguous — keep, flag) and **1,778 with different numbers** (likely label noise — flag, keep out of training positives, sample for the hand-labeled test set). The 544 rows with NULL `address_b` fall in neither count (NULL fails every comparison).

## 6. Address cleaning — `ct_positives_clean`

Needed for the **building-level** match definition: same street address = same location, units ignored.

**Decisions**
- Floors are stripped (inside the building, like a suite).
- `BLDG` / `BUILDING` is **kept** — a different building number counts as a different location.
- Road-number words are **not** designators: `ROUTE`/`RTE`/`RT`/`US`/`SR`/`CT`/`CONNECTICUT`/`I` (`ROUTE 1`, `I 95`). `BOX` (PO box) not stripped either.
- Punctuation is replaced with a **space**, not deleted (`16-18`, `RD,SUITE` must not glue together). `-` is kept.

**Designator list** — found by counting the word before a digit-containing last token (`STE` 1,656, `UNIT` 262, `SUITE` 143, `FL` 82, `SPC` 36, …), filtered by hand:
`SUITE STE UITE UNIT APARTMENT APT FLOOR FL SPACE SPC ROOM RM OFFICE OFC STORE PMB NUMBER PAD BAY`

The two cells below are the exploration behind two of the decisions; 6c onward is the pipeline.

### 6a. Units with no designator and no `#`

Does `… RD 2` / `… ST D16` happen often enough to need a rule? Last tokens after `RD`/`ST` are mostly directionals (`… MAIN ST N`), which must **not** be stripped. Only 52 bare unit codes → known gap.

### 6b. Two-word designators

`SPACE NUMBER 1029A`, `SUITE STE 3`: the single-designator pattern left the first word behind. 11 rows → the step-2 pattern allows an optional first designator.

### 6c. Cleaning chain + gap check

Order: floor → designator → `#` → punctuation → whitespace. Unit stripping must run **before** punctuation removal (it needs `#` and `-`).

Gap check = search the cleaned output for anything that should be gone. Remaining (both sides, out of 37,680), all accepted as **known gaps**:

| Leftover | A | B | Typical shape |
|---|---|---|---|
| designator word | 22 | 77 | two units (`SUITE 240 STE 700R`), code with a space (`SUITE G 110`), unit before `BLDG`, false alarms (`POST OFFICE SQ`) |
| city/state/ZIP in address | 44 | 101 | `… BRIDGEPORT CT 06607` |
| floor | 8 | 3 | `GROUND FLOOR`, `UPPER FLOOR`, `2ND STE 7 FLOOR` |
| `#` mid-address (before step 4) | 21 | 21 | `UNIT #1 & #2` |

Also known: 52 bare unit codes (6a), a raw newline inside one address (`233 ELM STREET\n2ND FLOOR`), free-text descriptions instead of addresses.

### 6d. Save — `ct_positives_clean_2026-09-23.1.parquet`

18 columns: the 15 from `ct_positives` + `address_a_clean`, `address_b_clean`, `same_street_address` (cleaned strings equal; NULL when `address_b` is NULL).

In [ ]:
# 6d. RUN ONCE (skips itself if the output file exists) — same chain as 6c, written out with all 15 original columns + the cleaned addresses + flag
# Keep the cleaning steps identical to 6c if either cell changes
out_path = f'Data/ct_positives_clean_{OVERTURE_RELEASE}.parquet'
if os.path.exists(out_path):
    print(f"skip: {out_path} already exists")
else:
    con.sql(rf"""
    COPY
    (WITH
        step1 AS (
                SELECT 
                *,
                regexp_replace(address_a, '\s+[A-Za-z0-9]+\s+(FLOOR|FL)$', '') AS a_remove_fl,
                regexp_replace(address_b, '\s+[A-Za-z0-9]+\s+(FLOOR|FL)$', '') AS b_remove_fl
                FROM read_parquet('Data/ct_positives_{OVERTURE_RELEASE}.parquet')
            ),
        step2 AS (
            SELECT
                *,
                regexp_replace(a_remove_fl, 
                    '\s+((SUITE|STE|UITE|UNIT|APARTMENT|APT|FLOOR|FL|SPACE|SPC|ROOM|RM|OFFICE|OFC|STORE|PMB|NUMBER|PAD|BAY)\s)?(SUITE|STE|UITE|UNIT|APARTMENT|APT|FLOOR|FL|SPACE|SPC|ROOM|RM|OFFICE|OFC|STORE|PMB|NUMBER|PAD|BAY)\s[#a-zA-Z0-9-]+$',
                    ''
                ) AS a_remove_des,
                regexp_replace(b_remove_fl, 
                    '\s+((SUITE|STE|UITE|UNIT|APARTMENT|APT|FLOOR|FL|SPACE|SPC|ROOM|RM|OFFICE|OFC|STORE|PMB|NUMBER|PAD|BAY)\s)?(SUITE|STE|UITE|UNIT|APARTMENT|APT|FLOOR|FL|SPACE|SPC|ROOM|RM|OFFICE|OFC|STORE|PMB|NUMBER|PAD|BAY)\s[#a-zA-Z0-9-]+$',
                    ''
                ) AS b_remove_des
            FROM step1
        ),
        step3 AS (
            SELECT
            *,
            regexp_replace(a_remove_des, '\s*\#\s*[A-Z0-9]+$', '') AS a_remove_hash,
            regexp_replace(b_remove_des, '\s*\#\s*[A-Z0-9]+$', '') AS b_remove_hash,
            FROM step2
        ),
        step4 AS (
            SELECT 
            *,
            regexp_replace(a_remove_hash, '[^A-Z0-9- ]', ' ', 'g') AS a_remove_punctuation,
            regexp_replace(b_remove_hash, '[^A-Z0-9- ]', ' ', 'g') AS b_remove_punctuation,
            FROM step3
        ),
        step5 AS (
            SELECT
            *, 
            TRIM(regexp_replace(a_remove_punctuation, '\s+', ' ', 'g')) AS a_remove_spaces,
            TRIM(regexp_replace(b_remove_punctuation, '\s+', ' ', 'g')) AS b_remove_spaces,
            FROM step4
        )
    SELECT
        gers_id,
        fsq_place_id,
        address_a,
        address_b,
        locality_a,
        locality_b,
        region_a,
        region_b,
        postcode_a,
        postcode_b,
        name_a,
        name_b,
        published_from,
        label, 
        identical_address,
        a_remove_spaces AS address_a_clean,
        b_remove_spaces AS address_b_clean,
        address_a_clean = address_b_clean AS same_street_address,
    FROM step5
    ) TO 'Data/ct_positives_clean_{OVERTURE_RELEASE}.parquet' (FORMAT parquet)
    """
    )

### 6e. Checks

Result: 37,680 rows, 544 NULL. **TF = 0** (nothing broke), TT = 26,286, **FT = 3,516**.

- Same street address: **29,802** (79%) vs. 26,286 (70%) identical raw strings.
- Of the 10,850 non-identical pairs with both addresses present, **32%** differed only by formatting/units.
- **7,334** positives still differ after cleaning — abbreviations, typos, different house numbers, city/ZIP in the address. These are the hard positives for the model.

## 7. House-number flags — `ct_positives_flagged`

- `number_mismatch`: both sides have a leading house number and they differ (likely Overture label noise) — **1,778**
- `number_missing`: exactly one side has a leading number — **999**
- Drops NULL `address_b` (544) and empty-string `address_a_clean` (46) → **37,090 rows**

Flags use the `_clean` columns. `regexp_extract` returns `''` (not NULL) when nothing matches.

In [ ]:
con.sql(rf"""
COPY
(SELECT
    *,
    -- both sides have a leading number, and they differ
    ((regexp_extract(address_a_clean, '^\d+') <> regexp_extract(address_b_clean, '^\d+'))
    AND regexp_extract(address_a_clean, '^\d+') <> ''
    AND regexp_extract(address_b_clean, '^\d+') <> '')
    AS number_mismatch,

    -- exactly one side has a leading number
    ((regexp_extract(address_a_clean, '^\d+') = '' OR regexp_extract(address_b_clean, '^\d+') = '') AND
    regexp_extract(address_a_clean, '^\d+') <> regexp_extract(address_b_clean, '^\d+')
    )
    AS number_missing
FROM
    read_parquet('Data/ct_positives_clean_{OVERTURE_RELEASE}.parquet')
WHERE
    address_b IS NOT NULL AND address_a_clean <> ''
) TO 'Data/ct_positives_flagged_{OVERTURE_RELEASE}.parquet' (FORMAT parquet)
""")

## 8. ZIP code health check

Before splitting by ZIP: `postcode_a` (the split key) has 10 NULL, 2 empty and **11,066 ZIP+4** values (~30%);
`postcode_b` has 1 NULL and 19 ZIP+4. 15 non-empty `postcode_a` values don't start with `06` — 12 of them are
4-digit ZIPs that lost their leading zero (e.g. `6371`), 3 are junk / out of state.

## 9. ZIP cleaning → `zip_5` and `zip_b` — `ct_positives_zip_cleaned`

`LPAD(postcode_a, 5, '0')` does both fixes at once: it pads the 4-digit ZIPs (`6371` → `06371`) and, because DuckDB's
`LPAD` truncates longer strings, cuts ZIP+4 down to 5 digits (`06514-4033` → `06514`). The raw `postcode_a` is kept.
All 12 padded values were checked against `postcode_b` and agree.

Result: **37,075 rows** (15 dropped: 10 NULL, 2 empty, 3 non-CT); **8,931 raw `postcode_a` values → 331 ZIPs**.

`zip_b` is side B's (Foursquare) ZIP, carried as a **feature**, not a filter — no rows are dropped
for it. Same `LPAD` trick (pads 4-digit, truncates ZIP+4), but kept only if the result is exactly 5 digits, so junk
(one `postcode_b` is a phone number, `203-874-6772`) becomes NULL. Out-of-state values are kept (real source
disagreement). Checked: 37,075 rows, 2 NULL, 0 values that aren't 5 digits.

In [ ]:
con.sql(rf"""
COPY(
SELECT
    *,
    LPAD(postcode_a, 5, '0') as zip_5,   -- pads 4-digit ZIPs AND truncates ZIP+4 (intentional)
    CASE                                                             -- side B: same LPAD, but junk -> NULL
        WHEN regexp_matches(LPAD(postcode_b, 5, '0'), '^\d{{5}}$')
            THEN LPAD(postcode_b, 5, '0')
        ELSE NULL
    END as zip_b
FROM
    read_parquet('Data/ct_positives_flagged_{OVERTURE_RELEASE}.parquet')
WHERE
    postcode_a IS NOT NULL AND
    postcode_a <> '' AND
    (regexp_matches(postcode_a, '^06') OR                              -- CT ZIPs
    (LENGTH(postcode_a) = 4 AND regexp_matches(postcode_a, '^6'))      -- CT ZIPs missing their leading zero
    )
) TO 'Data/ct_positives_zip_cleaned_{OVERTURE_RELEASE}.parquet' (FORMAT parquet )
""")